# Competitive race (B — prelims)

Before you start working with this notebook, remember to:

* Ensure the value of `max_time` is what you want.

If you interrupt a cell and the simulator stops responding, you can try creating a new simulator (and new views).

Remember that it may take a long time to find a set of prelim races that results in two finalists per team. Remember that it then takes a long time to make videos for that set of prelim races.

Specify maximum length of each race (in seconds) before time-out.

In [ ]:
max_time = 120.

Import modules and configure the notebook.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import secrets
import json
import shutil
from pathlib import Path
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_agg import FigureCanvasAgg
import ae353_drone

Prevent students from importing `ae353_drone` in their own code.

In [ ]:
import sys
sys.modules['ae353_drone'] = None

Treat `RuntimeWarnings` as errors so that student code does not completely break the simulator (e.g., making it so collision checking stops working and drones fly through the ground).

In [ ]:
import warnings
warnings.filterwarnings('error', category=RuntimeWarning)

Create and print seed so it is possible to reproduce the results.

In [ ]:
seed = secrets.randbits(32)
print(seed)

Create simulator.

In [ ]:
simulator = ae353_drone.Simulator(seed=seed)

Add camera view (only one is needed for prelims).

In [ ]:
simulator.add_view('my_start_view', 'start')

Load information about qualification race.

In [ ]:
race_information_path = Path('race-information.json')
with open(race_information_path, 'r') as infile:
    information = json.load(infile)
datetimestr = information['datetimestr']
teams = information['teams']
srcdir_designs = Path(f'{datetimestr}-designs')

Define functions to get students by netid and partners by student.

In [ ]:
def get_student(students, netid):
    for student in students:
        if student['netid'] == netid:
            return student
    return None

def get_partners(students, student):
    partner_netids = np.array(student['dp4_partner']).flatten().tolist()
    partner_students = []
    for netid in partner_netids:
        partner_students.append(get_student(students, netid))
    return partner_students

Define functions to show results.

In [ ]:
benign_failures = [
    'Inactive.',
    'Out of bounds.',
]

def get_netids_to_email(drone_name, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    
    netids_to_email = []
    netids_to_email.append(student['netid'] + '@illinois.edu')
    partners = get_partners(students, student)
    for partner in partners:
        netids_to_email.append(partner['netid'] + '@illinois.edu')
    
    return netids_to_email

def get_student_name(drone_name, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    
    name = f'{student["first_name"]} {student["last_name"]}'
    partners = get_partners(students, student)
    for partner in partners:
        name += f' and {partner["first_name"]} {partner["last_name"]}'
        
    return name

def disqualify_student(drone_name, drone_error, students):
    student = get_student(students, drone_name)
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone_name}')
    student['dp4_status'] = 'disqualified'
    student['dp4_error'] = drone_error
    partners = get_partners(students, student)
    for partner in partners:
        partner['dp4_status'] = 'disqualified'
        partner['dp4_error'] = drone_error

def get_results(simulator, students):
    netids_to_email = []
    finished = []
    still_running = []
    failed = []
    errors = ''
    results = ''
    for drone in simulator.drones:
        if drone['finish_time'] is not None:
            finished.append((drone, drone['finish_time']))
        elif drone['running']:
            still_running.append(drone)
        else:
            failed.append(drone)
            errors += f'======================\n{drone["error"]}\n======================\n\n'
    finished = sorted(finished, key=lambda f: f[1])
    
    results += 'FINISHED\n'
    for d in finished:
        drone = d[0]
        drone_name = drone['name']
        student_name = get_student_name(drone_name, students)
        results += f' {d[1]:6.2f} : {drone_name:20s} : {student_name}\n'

    results += '\nSTILL RUNNING\n'
    for d in still_running:
        drone = d
        drone_name = drone['name']
        student_name = get_student_name(drone_name, students)
        results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nINACTIVE OR OUT OF BOUNDS\n'
    for d in failed:
        drone = d
        drone_name = drone['name']
        if drone['error'] in benign_failures:
            student_name = get_student_name(drone_name, students)
            results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nFAILED\n'
    for d in failed:
        drone = d
        drone_name = drone['name']
        if drone['error'] not in benign_failures:
            disqualify_student(drone_name, drone['error'], students)
            student_name = get_student_name(drone_name, students)
            netids_to_email.extend(get_netids_to_email(drone_name, students))
            results += f'        : {drone_name:20s} : {student_name}\n'
    
    results += '\nERRORS (REASONS FOR FAILURE)\n\n'
    results += errors
    
    results += '\nNETIDS TO EMAIL ABOUT FAILURE\n\n'
    results += (' ' + ', '.join(netids_to_email))
    
    return results

Define functions to run prelim races.

In [ ]:
def get_team(racer):
    student = get_student(students, racer)
    assert(student is not None)
    return student['dp4_team']

def get_racer_one(racers):
    racer_one = racers[0]
    racers.remove(racer_one)
    return racer_one, get_team(racer_one)

def get_racer_two(racers, team):
    for racer_two in racers:
        if team == get_team(racer_two):
            racers.remove(racer_two)
            return racer_two
    return None

def count_on_team(racers, team):
    count = 0
    for racer in racers:
        if get_team(racer) == team:
            count += 1
    return count

def get_racers_on_team(racers, team):
    racers_on_team = []
    for racer in racers:
        if get_team(racer) == team:
            racers_on_team.append(racer)
    return racers_on_team

def get_racer_label(students, racer):
    student = get_student(students, racer)
    name = f'{student["first_name"]} {student["last_name"]}'
    partners = get_partners(students, student)
    for partner in partners:
        name += f'\n{partner["first_name"]} {partner["last_name"]}'
    team = student['dp4_team']
    name += f'\n\nTEAM {team.upper()}'
    return name, teams[team]

def get_racer_status(drone):
    if drone['finish_time'] is not None:
        return f'FINISHED ({drone['finish_time']:.2f})'
    if drone['running']:
        return 'TIMED OUT'
    return 'FAILED'

def get_winning_racer(drones):
    winning_name = None
    winning_time = np.inf
    for drone in drones:
        if drone['finish_time'] is None:
            continue
        if drone['finish_time'] < winning_time:
            winning_name = drone['name']
            winning_time = drone['finish_time']
    return winning_name

# From Google AI (search string: "matplotlib convert figure directly to rgba array")
def fig_to_rgba_array(fig):
    """
    Convert a Matplotlib figure to a 4D NumPy array with RGBA channels.

    Args:
        fig (matplotlib.figure.Figure): The Matplotlib figure to convert.

    Returns:
        numpy.ndarray: A NumPy array representing the figure's image data
                       in RGBA format (height, width, 4).
    """
    # Attach a FigureCanvasAgg to the figure if it doesn't have one
    if not hasattr(fig, 'canvas') or fig.canvas is None:
        canvas = FigureCanvasAgg(fig)
    else:
        canvas = fig.canvas

    # Draw the renderer to ensure the figure is rendered
    canvas.draw()

    # Get the RGBA buffer from the figure
    # Note: tostring_argb() returns ARGB, so we need to reorder
    w, h = canvas.get_width_height()
    buf = np.frombuffer(canvas.tostring_argb(), dtype=np.uint8)
    buf.shape = (h, w, 4)

    # Roll the ALPHA channel to have it in RGBA mode
    # ARGB (Alpha, Red, Green, Blue) becomes RGBA
    buf = np.roll(buf, 3, axis=2)
    return buf

def _create_title_frame(drones, students, finish=False, width=640, height=480, dpi=100):
    if len(drones) != 2:
        raise Exception('There must be exactly two drones in the race.')

    drone_one = drones[0]
    drone_two = drones[1]
    racer_one = drone_one['name']
    racer_two = drone_two['name']
    path_png_one = Path(drone_one['image'])
    path_png_two = Path(drone_two['image'])

    width = 640
    height = 480
    dpi = 100

    dx = 40
    y = 200
    w = 200

    fig = plt.figure(figsize=(width/dpi, height/dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xticks([])
    ax.set_yticks([])
    # ax.set_frame_on(False)
    ax.set_xlim([0, width])
    ax.set_ylim([0, height])
    ax.set_aspect('equal')
    im = plt.imread(path_png_one)
    ax.imshow(im, aspect='equal', extent=(dx, dx + w, y, y + w))
    label_one, rgb_one = get_racer_label(students, racer_one)
    ax.text(dx + (w / 2), y - dx, label_one, ha='center', va='top', fontsize=12)
    ax.add_patch(Rectangle((dx + (w / 2) - (dx / 2), dx / 2), dx, dx, linewidth=0, facecolor=rgb_one))
    im = plt.imread(path_png_two)
    ax.imshow(im, aspect='equal', extent=(width - (w + dx), width - dx, y, y + w))
    label_two, rgb_two = get_racer_label(students, racer_two)
    ax.text(width - (dx + (w / 2)), y - dx, label_two, ha='center', va='top', fontsize=12)
    ax.add_patch(Rectangle((width - (dx + (w / 2)) - (dx / 2), dx / 2), dx, dx, linewidth=0, facecolor=rgb_two))
    ax.text(width / 2, y + w / 2, 'versus', ha='center', va='center', fontsize=18)

    if finish:
        ax.text(
            dx + (w / 2), y + w + (dx / 4),
            get_racer_status(drone_one),
            ha='center', va='bottom', fontsize=12,
        )
        ax.text(
            width - (dx + (w / 2)), y + w + (dx / 4),
            get_racer_status(drone_two),
            ha='center', va='bottom', fontsize=12,
        )
        winning_racer = get_winning_racer(drones)
        if winning_racer == racer_one:
            ax.text(
                dx + (w / 2), height - dx,
                'WINNER',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
        elif winning_racer == racer_two:
            ax.text(
                width - (dx + (w / 2)), height - dx,
                'WINNER',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
        else:
            ax.text(
                width / 2, height - dx,
                'NO WINNER (VOID RACE)',
                ha='center', va='bottom', fontsize=16, weight='bold',
            )
    
    return fig_to_rgba_array(fig)

def run_race(racer_one, racer_two, srcdir, dstdir, students, make_videos=False):
    # Get files ready
    os.mkdir(dstdir)
    for racer in [racer_one, racer_two]:
        shutil.copyfile(os.path.join(srcdir, f'{racer}.py'), os.path.join(dstdir, f'{racer}.py'))
        shutil.copyfile(os.path.join(srcdir, f'{racer}.png'), os.path.join(dstdir, f'{racer}.png'))

    # Get simulator ready
    simulator.clear_drones()
    simulator.place_rings()
    simulator.load_drones(dstdir)
    simulator.reset()

    # Get callbacks ready
    create_start_frame = lambda drones: _create_title_frame(drones, students, finish=False)
    create_finish_frame = lambda drones: _create_title_frame(drones, students, finish=True)

    # Run simulation
    if make_videos:
        simulator.enable_views()
        simulator.run(
            max_time=max_time,
            print_debug=True,
            videos=[
                {
                    'view_name': 'my_start_view',
                    'file_name': f'{dstdir}.mp4',
                    'start_frame': {
                        'callback': create_start_frame,
                        'length': 1,
                        'pause': 1,
                    },
                    'finish_frame': {
                        'callback': create_finish_frame,
                        'length': 1,
                        'pause': 1,
                    },
                },
            ],
        )
    else:
        simulator.disable_views()
        simulator.run(max_time=max_time, print_debug=False)

    # Find winner
    winning_name = None
    winning_time = np.inf
    for drone in simulator.drones:
        if drone['finish_time'] is None:
            continue
        if drone['finish_time'] < winning_time:
            winning_name = drone['name']
            winning_time = drone['finish_time']

    # Get and write results (also, update students to reflect disqualifications)
    results = get_results(simulator, students)
    with open(f'{dstdir}.txt', 'w') as f:
        f.write(results)

    return winning_name

Initialize number of attempts.

In [ ]:
number_of_attempts = 0

## Run prelim races without videos

Keep running complete sets of prelim races until you end up with two finalists from each team.

In [ ]:
while True:

    ##########################
    # PREPARE TO RUN RACES
    ##########################

    # Increment number of attempts
    number_of_attempts += 1
    print(f'\nATTEMPT {number_of_attempts}\n')

    # Get state of random number generator
    rng_state = simulator.rng.bit_generator.state

    # Get student roster (updated with qualification failures)
    with open(Path(f'{datetimestr}-A-students.json'), 'r') as infile:
        students = json.load(infile)

    # Get list of students who qualified
    with open(Path(f'{datetimestr}-A-qualified.json'), 'r') as infile:
        qualified = json.load(infile)

    # Shuffle list of students who qualified
    simulator.rng.shuffle(qualified)

    # Delete all prior race directories and files, if they exist
    for item in Path('.').iterdir():
        if item.name.startswith(f'{datetimestr}-B-race'):
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
            else:
                raise Exception(f'{item.name} is neither a file nor a directory')

    ##########################
    # RUN RACES
    ##########################

    index_of_race = 0
    results = ''
    finalists = []

    for team in teams.keys():
        # If we have already failed, stop trying
        if None in finalists:
            continue

        # Start with all racers on team
        racers = get_racers_on_team(qualified, team)
        racers.extend([None] * (16 - len(racers)))
        current_round = np.array(racers).reshape((2, -1)).T.tolist()

        # Initialize text to store results
        results += f'\n========================\nTEAM {team.upper()}\n\n'

        print(f'\n**** TEAM {team.upper()} ****')
        for round_of in [8, 4, 2]:
            print(f'\n========================\nROUND OF {2 * round_of}\n========================\n')
            results += f'ROUND OF {2 * round_of}\n'
            next_round = [[] for i in range(int(round_of / 2))]
            for i, (racer_one, racer_two) in enumerate(current_round):
                
                if (racer_one is None) and (racer_two is None):
                    next_round[int(np.floor(i / 2))].append(None)
                    result = f'     : {"_______________":15s} {"_______________":15s} > {"_______________":15s}\n'
                    print(result)
                    results += result
                    continue

                if (racer_one is None):
                    next_round[int(np.floor(i / 2))].append(racer_two)
                    result = f'     : {"_______________":15s} {racer_two:15s} > {racer_two:15s}\n'
                    print(result)
                    results += result
                    continue

                if (racer_two is None):
                    next_round[int(np.floor(i / 2))].append(racer_one)
                    result = f'     : {racer_one:15s} {"_______________":15s} > {racer_one:15s}\n'
                    print(result)
                    results += result
                    continue
                
                index_of_race += 1
                winning_racer = run_race(
                    racer_one, racer_two,
                    srcdir_designs,
                    f'{datetimestr}-B-race-{index_of_race:03d}-{team}-{racer_one}-{racer_two}',
                    students,
                    make_videos=False,
                )

                if winning_racer is None:
                    result = f' {index_of_race:03d} : {racer_one:15s} {racer_two:15s} > {"_______________":15s}\n'
                else:
                    result = f' {index_of_race:03d} : {racer_one:15s} {racer_two:15s} > {winning_racer:15s}\n'
                print(result)
                results += result
                
                next_round[int(np.floor(i / 2))].append(winning_racer)
            
            results += '\n'
            current_round = next_round
        
        finalists.extend(current_round[0])
    
    ##########################
    # CHECK FOR GODO RESULT
    ##########################

    if not (None in finalists):
        break

## Run (identical) prelim races with videos

Prepare to run races.

In [ ]:
# Restore state of random number generator
simulator.rng.bit_generator.state = rng_state

# Get student roster (updated with qualification failures)
with open(Path(f'{datetimestr}-A-students.json'), 'r') as infile:
    students = json.load(infile)

# Get list of students who qualified
with open(Path(f'{datetimestr}-A-qualified.json'), 'r') as infile:
    qualified = json.load(infile)

# Shuffle list of students who qualified
simulator.rng.shuffle(qualified)

# Delete all prior race directories and files, if they exist
for item in Path('.').iterdir():
    if item.name.startswith(f'{datetimestr}-B-race'):
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)
        else:
            raise Exception(f'{item.name} is neither a file nor a directory')

Turn interactive plotting off.

In [ ]:
plt.ioff()

Run races.

In [ ]:
index_of_race = 0
results = ''
finalists = []

for team in teams.keys():

    # Start with all racers on team
    racers = get_racers_on_team(qualified, team)
    racers.extend([None] * (16 - len(racers)))
    current_round = np.array(racers).reshape((2, -1)).T.tolist()

    # Initialize text to store results
    results += f'\n========================\nTEAM {team.upper()}\n\n'

    print(f'\n**** TEAM {team.upper()} ****')
    for round_of in [8, 4, 2]:
        print(f'\n========================\nROUND OF {2 * round_of}\n========================\n')
        results += f'ROUND OF {2 * round_of}\n'
        next_round = [[] for i in range(int(round_of / 2))]
        for i, (racer_one, racer_two) in enumerate(current_round):
            
            if (racer_one is None) and (racer_two is None):
                next_round[int(np.floor(i / 2))].append(None)
                result = f'     : {"_______________":15s} {"_______________":15s} > {"_______________":15s}\n'
                print(result)
                results += result
                continue

            if (racer_one is None):
                next_round[int(np.floor(i / 2))].append(racer_two)
                result = f'     : {"_______________":15s} {racer_two:15s} > {racer_two:15s}\n'
                print(result)
                results += result
                continue

            if (racer_two is None):
                next_round[int(np.floor(i / 2))].append(racer_one)
                result = f'     : {racer_one:15s} {"_______________":15s} > {racer_one:15s}\n'
                print(result)
                results += result
                continue
            
            index_of_race += 1
            winning_racer = run_race(
                racer_one, racer_two,
                srcdir_designs,
                f'{datetimestr}-B-race-{index_of_race:03d}-{team}-{racer_one}-{racer_two}',
                students,
                make_videos=True,
            )

            if winning_racer is None:
                result = f' {index_of_race:03d} : {racer_one:15s} {racer_two:15s} > {"_______________":15s}\n'
            else:
                result = f' {index_of_race:03d} : {racer_one:15s} {racer_two:15s} > {winning_racer:15s}\n'
            print(result)
            results += result
            
            next_round[int(np.floor(i / 2))].append(winning_racer)
        
        results += '\n'
        current_round = next_round
        
    finalists.extend(current_round[0])

Turn interactive plotting on.

In [ ]:
plt.ion()

Print results (look at this for the brackets). These results will also be written to a file (see below).

In [ ]:
print(results)

Save results of prelims to file.

In [ ]:
# Student roster (updated with prelim failures)
with open(f'{datetimestr}-B-students.json', 'w') as outfile:
    json.dump(students, outfile, indent=4)

# List of finalists
qualified = [drone['name'] for drone in simulator.drones]
with open(f'{datetimestr}-B-finalists.json', 'w') as outfile:
    json.dump({'finalists': finalists}, outfile, indent=4)

# Information about race (updated with prelim information)
information['seed-B'] = seed
information['iters-B'] = number_of_attempts
with open(race_information_path, 'w') as outfile:
    json.dump(information, outfile, indent=4,)

# Write results (look at this for the brackets)
with open(f'{datetimestr}-B-results.txt', 'w') as f:
    f.write(results)